# 0.1 Import Libraries

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# 0.2 Load Project Modules

In [ ]:
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from config.settings import PROCESSED_DIR, OUTPUT_DIR, DEFAULT_RANDOM_STATE
from src.missing_retailer import holdout_random_stores, holdout_top_contributing_stores, holdout_stores_by_group, estimate_missing_sales_similar_store_average, estimate_missing_sales_historical_trend, estimate_missing_sales_category_trend, compare_panel_recovery_with_imputation, compare_market_estimate_with_imputation

# 1.1 Load Panel and Universe Data

In [ ]:
universe = pd.read_csv(OUTPUT_DIR / "active_store_universe.csv")
panel_store_list = pd.read_csv(OUTPUT_DIR / "sample_panel_store_list.csv")
panel = panel_store_list[panel_store_list["panel_name"] == "optimized_panel"].copy()
weekly_sales = pd.read_csv(PROCESSED_DIR / "weekly_store_sales.csv", parse_dates=["week"])
weekly_category_sales = pd.read_csv(PROCESSED_DIR / "weekly_store_category_sales.csv", parse_dates=["week"])
panel.head()

# 1.2 Define Missing Retailer Scenarios

In [ ]:
holdout_count = max(1, round(len(panel) * 0.15))
cluster_value = panel.groupby("cluster")["store_nbr"].nunique().sort_values(ascending=False).index[0]
scenarios = {}

# 2.1 Hold Out Random Stores

In [ ]:
scenarios["random_store_holdout"] = holdout_random_stores(panel, holdout_count, DEFAULT_RANDOM_STATE)
scenarios["random_store_holdout"][["store_nbr", "total_sales"]].head()

# 2.2 Hold Out Top-Contributing Stores

In [ ]:
scenarios["top_store_holdout"] = holdout_top_contributing_stores(panel, holdout_count)
scenarios["top_store_holdout"][["store_nbr", "total_sales"]].head()

# 2.3 Hold Out Store Cluster

In [ ]:
scenarios["cluster_holdout"] = holdout_stores_by_group(panel, "cluster", cluster_value)
scenarios["cluster_holdout"][["store_nbr", "cluster"]].head()

# 3.1 Estimate Missing Sales by Similar Stores

In [ ]:
similar_store_imputations = {
    name: estimate_missing_sales_similar_store_average(weekly_sales, panel, heldout)
    for name, heldout in scenarios.items()
}
next(iter(similar_store_imputations.values())).head()

# 3.2 Estimate Missing Sales by Historical Trend

In [ ]:
historical_imputations = {
    name: estimate_missing_sales_historical_trend(weekly_sales, heldout)
    for name, heldout in scenarios.items()
}
next(iter(historical_imputations.values())).head()

# 3.3 Estimate Missing Sales by Category Trend

In [ ]:
category_imputations = {
    name: estimate_missing_sales_category_trend(weekly_category_sales, panel, heldout)
    for name, heldout in scenarios.items()
}
next(iter(category_imputations.values())).head()

# 4.1 Compare Impact Without Imputation

In [ ]:
rows = []
panel_sales = weekly_sales[weekly_sales["store_nbr"].isin(panel["store_nbr"])]
for name, heldout in scenarios.items():
    rows.append(compare_panel_recovery_with_imputation(name, panel_sales, heldout, None, "without_imputation"))
    rows.append(compare_market_estimate_with_imputation(name, weekly_sales, panel_sales, heldout, None, "without_imputation"))
impact_without_imputation = pd.DataFrame(rows)
impact_without_imputation

# 4.2 Compare Impact With Imputation

In [ ]:
rows = []
panel_sales = weekly_sales[weekly_sales["store_nbr"].isin(panel["store_nbr"])]
imputation_sets = {
    "similar_store_average": similar_store_imputations,
    "historical_trend": historical_imputations,
    "category_trend": category_imputations,
}
for name, heldout in scenarios.items():
    for method, imputations in imputation_sets.items():
        rows.append(compare_panel_recovery_with_imputation(name, panel_sales, heldout, imputations[name], method))
        rows.append(compare_market_estimate_with_imputation(name, weekly_sales, panel_sales, heldout, imputations[name], method))
impact_with_imputation = pd.DataFrame(rows)
impact_with_imputation.head()

# 4.3 Save Missing Retailer Outputs

In [ ]:
missing_retailer_impact = pd.concat([impact_without_imputation, impact_with_imputation], ignore_index=True)
missing_retailer_impact.to_csv(OUTPUT_DIR / "missing_retailer_impact_summary.csv", index=False)
missing_retailer_impact